# Burstchester Direct Script Colab Test Training

이 노트북은 `https://github.com/tomongoose/burstchester-cli`에서 Python trainer 스크립트만 가져와 Burstchester dataset ID로 모델 훈련을 테스트한다.

필요한 준비물:

- Burstchester 웹에서 발급한 CLI access token
- 훈련에 사용할 Burstchester dataset ID 1개 이상
- 선택 사항: gated Hugging Face 모델 또는 모델 업로드에 사용할 `HF_TOKEN`

Colab 런타임은 `Runtime > Change runtime type > GPU`로 설정하는 것을 권장한다.

In [ ]:
#@title 1. Training settings
REPO_URL = "https://github.com/tomongoose/burstchester-cli.git" #@param {type:"string"}
REPO_BRANCH = "gemma4-full-trainer" #@param {type:"string"}
REPO_DIR = "/content/burstchester-cli" #@param {type:"string"}
DATASET_IDS = "legal-ko" #@param {type:"string"}
DATASET_DOWNLOAD_URL = "https://us-central1-bustchester-e08c3.cloudfunctions.net/prepareDatasetDownload" #@param {type:"string"}
MODEL_REGISTER_URL = "https://us-central1-bustchester-e08c3.cloudfunctions.net/registerModelHttp" #@param {type:"string"}
BASE_MODEL = "google/gemma-4-E2B" #@param {type:"string"}
TRAIN_COMMAND = "train-gemma4-e2b-full" #@param ["train", "train-gemma-2b-it-lora", "train-gemma4-e2b-full"]
TRAINING_METHOD = "full" #@param ["qlora", "lora", "full"]
WORKSPACE = "/content/burstchester-training" #@param {type:"string"}
EPOCHS = "1" #@param {type:"string"}
BATCH_SIZE = "1" #@param {type:"string"}
MAX_SEQ_LENGTH = "128" #@param {type:"string"}
LORA_RANK = "8" #@param {type:"string"}
LORA_ALPHA = "16" #@param {type:"string"}
LORA_DROPOUT = "0.05" #@param {type:"string"}
SKIP_REGISTER = True #@param {type:"boolean"}
OUTPUT_MODEL_REPO = "" #@param {type:"string"}
MODEL_POINT_COST = "100" #@param {type:"string"}

print("Settings loaded. Secrets are requested in the next cell.")

In [ ]:
# 2. Load secrets without saving them in the notebook output.
import os
from getpass import getpass

try:
    from google.colab import userdata
except ImportError:
    userdata = None

def get_colab_secret(name):
    if userdata is None:
        return ""
    try:
        return userdata.get(name) or ""
    except Exception:
        return ""

if not DATASET_IDS.strip():
    raise ValueError("Set DATASET_IDS before continuing.")

access_token = os.environ.get("BURSTCHESTER_ACCESS_TOKEN") or get_colab_secret("BURSTCHESTER_ACCESS_TOKEN")
if not access_token:
    access_token = getpass("Burstchester CLI access token: ")
if not access_token.strip():
    raise ValueError("Set BURSTCHESTER_ACCESS_TOKEN in Colab Secrets or enter it when prompted.")
os.environ["BURSTCHESTER_ACCESS_TOKEN"] = access_token.strip()

hf_token = os.environ.get("HF_TOKEN") or get_colab_secret("HF_TOKEN")
if not hf_token:
    entered = getpass("Hugging Face token (optional, press Enter to skip): ")
    hf_token = entered.strip()
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

print("Secrets configured in environment.")

In [ ]:
# 3. Clone or update the trainer script repository.
import os
import subprocess

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

subprocess.run(["git", "status", "--short", "--branch"], cwd=REPO_DIR, check=True)

In [ ]:
# 4. Install Python training dependencies.
# Keep Colab's preinstalled torch/CUDA stack intact. Upgrading torch can
# pull a different CUDA toolkit and conflict with torchvision/torchaudio/RAPIDS.
!python -m pip install -q -U transformers peft accelerate datasets huggingface_hub
!python -m pip install -q -U --no-deps bitsandbytes

if TRAIN_COMMAND == "train-gemma4-e2b-full":
    try:
        from transformers import AutoModelForImageTextToText, AutoProcessor
    except ImportError:
        raise RuntimeError(
            "Gemma 4 training requires transformers with AutoModelForImageTextToText. "
            "Restart the runtime and rerun the install cell after upgrading transformers."
        )


In [ ]:
# 5. Download and merge datasets directly through the Burstchester API.
from pathlib import Path
import json
import os
import re
import urllib.parse
import urllib.request
import zipfile

repo = Path(REPO_DIR)
workspace = Path(WORKSPACE)
workspace.mkdir(parents=True, exist_ok=True)
dataset_ids = list(dict.fromkeys([value for value in re.split(r"[\s,]+", DATASET_IDS.strip()) if value]))
dataset_file = Path(WORKSPACE).with_suffix(".dataset-ids.txt")
dataset_file.parent.mkdir(parents=True, exist_ok=True)
dataset_file.write_text("\n".join(dataset_ids) + "\n", encoding="utf-8")

def fetch_json(url, headers=None, data=None):
    request = urllib.request.Request(url, headers=headers or {}, data=data)
    with urllib.request.urlopen(request) as response:
        return json.loads(response.read().decode("utf-8"))

def dataset_metadata(dataset_id):
    url = DATASET_DOWNLOAD_URL + "?" + urllib.parse.urlencode({"datasetId": dataset_id})
    payload = fetch_json(url, headers={
        "Accept": "application/json",
        "Authorization": f"Bearer {os.environ['BURSTCHESTER_ACCESS_TOKEN']}",
    })
    if not payload.get("ok") or not payload.get("url"):
        raise RuntimeError(payload.get("error") or f"Dataset download failed for {dataset_id}")
    return payload

merged_dataset_path = workspace / "merged-dataset.jsonl"
with merged_dataset_path.open("w", encoding="utf-8") as merged:
    for dataset_id in dataset_ids:
        metadata = dataset_metadata(dataset_id)
        zip_path = workspace / f"{dataset_id}.zip"
        urllib.request.urlretrieve(metadata["url"], zip_path)

        dataset_dir = workspace / "datasets" / dataset_id
        dataset_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(dataset_dir)

        jsonl_path = dataset_dir / "dataset.jsonl"
        text = jsonl_path.read_text(encoding="utf-8")
        merged.write(text if text.endswith("\n") else text + "\n")

print("Merged datasets:", dataset_ids)
print("Merged dataset path:", merged_dataset_path)

In [ ]:
# 6. Build the trainer config JSON directly.
import json

effective_training_method = "full" if TRAIN_COMMAND == "train-gemma4-e2b-full" else TRAINING_METHOD
if TRAIN_COMMAND == "train-gemma-2b-it-lora":
    effective_training_method = "lora"

train_config = {
    "datasetId": dataset_ids[0],
    "datasetIds": dataset_ids,
    "datasetPath": str(merged_dataset_path),
    "modelRepo": BASE_MODEL,
    "outputDir": str(Path(WORKSPACE) / "output"),
    "trainingMethod": effective_training_method,
    "numTrainEpochs": float(EPOCHS),
    "perDeviceTrainBatchSize": int(BATCH_SIZE),
    "gradientAccumulationSteps": 8 if TRAIN_COMMAND == "train-gemma4-e2b-full" else 4,
    "learningRate": 0.00005 if TRAIN_COMMAND == "train-gemma4-e2b-full" else 0.0002,
    "maxSeqLength": int(MAX_SEQ_LENGTH),
    "loraRank": int(LORA_RANK),
    "loraAlpha": int(LORA_ALPHA),
    "loraDropout": float(LORA_DROPOUT),
    "loggingSteps": 10,
    "saveSteps": 100,
}

config_path = Path(WORKSPACE) / "train-config.json"
config_path.write_text(json.dumps(train_config, indent=2) + "\n", encoding="utf-8")
print(config_path.read_text(encoding="utf-8"))

In [ ]:
# 7. Run the Python trainer script directly.
import subprocess

trainer_scripts = {
    "train": repo / "src/python/train.py",
    "train-gemma-2b-it-lora": repo / "src/python/train_gemma_2b_it_lora.py",
    "train-gemma4-e2b-full": repo / "src/python/train_gemma4_e2b_full.py",
}
trainer_script = trainer_scripts[TRAIN_COMMAND]
train_cmd = ["python", str(trainer_script), "--config", str(config_path)]

print("Running:", " ".join(train_cmd))
subprocess.run(train_cmd, check=True)

In [ ]:
# 8. Optional: upload the trained output to Hugging Face and register it in Burstchester.
# Leave SKIP_REGISTER=True for a training-only smoke test.
from pathlib import Path
import json
import os
import urllib.request

if SKIP_REGISTER:
    print("Skipping upload/register. Trained output should be under:", Path(WORKSPACE) / "output")
else:
    if not OUTPUT_MODEL_REPO.strip():
        raise ValueError("Set OUTPUT_MODEL_REPO or set SKIP_REGISTER=True.")
    if not os.environ.get("HF_TOKEN"):
        raise ValueError("HF_TOKEN is required to upload to Hugging Face.")

    from huggingface_hub import HfApi, create_repo
    output_dir = str(Path(WORKSPACE) / "output")
    create_repo(repo_id=OUTPUT_MODEL_REPO, token=os.environ["HF_TOKEN"], exist_ok=True)
    HfApi(token=os.environ["HF_TOKEN"]).upload_folder(
        folder_path=output_dir,
        repo_id=OUTPUT_MODEL_REPO,
        repo_type="model",
    )
    model_url = f"https://huggingface.co/{OUTPUT_MODEL_REPO}"

    register_body = json.dumps({
        "huggingFaceUrl": model_url,
        "baseModel": BASE_MODEL,
        "trainingDatasets": dataset_ids,
        "trainingMethod": effective_training_method,
        "pointCost": MODEL_POINT_COST,
    }).encode("utf-8")
    register_request = urllib.request.Request(
        MODEL_REGISTER_URL,
        data=register_body,
        headers={
            "Authorization": f"Bearer {os.environ['BURSTCHESTER_ACCESS_TOKEN']}",
            "Content-Type": "application/json",
            "Accept": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(register_request) as response:
        print(response.read().decode("utf-8"))